In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score , root_mean_squared_error , mean_absolute_error
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import PowerTransformer
from xgboost import XGBRegressor
import optuna
import mlflow
import dagshub
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate , plot_slice

In [11]:
dagshub.init(repo_owner='mridul0010', repo_name='NYC-Taxi-Trip-Duration', mlflow=True)

Initialized MLflow to track repo "mridul0010/NYC-Taxi-Trip-Duration"

Repository mridul0010/NYC-Taxi-Trip-Duration initialized!

In [12]:
mlflow.set_tracking_uri("https://dagshub.com/mridul0010/NYC-Taxi-Trip-Duration.mlflow")

In [13]:
mlflow.set_experiment("5. HyperParameter Tuning OSRM - XGBoost")

<Experiment: artifact_location='mlflow-artifacts:/2c21882945d64610b5948897aefef0c5', creation_time=1784634131108, effective_trace_archival_retention=None, experiment_id='9', last_update_time=1784634131108, lifecycle_stage='active', name='5. HyperParameter Tuning OSRM - XGBoost', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [14]:
X_train = pd.read_csv("../data/processed/osrm_boosted/features.csv")
X_test  = pd.read_csv("../data/processed/osrm_boosted/features_test.csv")
y_train = pd.read_csv("../data/processed/osrm_boosted/labels.csv")
y_test = pd.read_csv("../data/processed/osrm_boosted/labels_test.csv")

In [15]:
print("Shape of X_train :-",X_train.shape)
print("Shape of y_train :-",y_train.shape)
print("Shape of X_test :-",X_test.shape)
print("Shape of y_test :-",y_test.shape)

Shape of X_train :- (1125732, 37)
Shape of y_train :- (1125732, 1)
Shape of X_test :- (281434, 37)
Shape of y_test :- (281434, 1)


In [16]:
pd.set_option('display.max_columns' , None)

In [17]:
def objective(trial):
    with mlflow.start_run(nested=True):
        mlflow.set_tag("model", "XGBoost")
        mlflow.set_tag("model_type", "Regressor")
        
        param = {
            "n_estimators": trial.suggest_int("n_estimators", 300, 800),
            "max_depth": trial.suggest_int("max_depth", 5, 12),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15 , log = True),
            "subsample": trial.suggest_float("subsample", 0.6, 0.95),
            "min_child_weight": trial.suggest_int("min_child_weight", 10, 50),
            "gamma": trial.suggest_float("gamma", 1e-3, 5.0 , log = True), 
            "reg_lambda": trial.suggest_float("reg_lambda", 1, 20),
            'colsample_bytree': trial.suggest_float("colsample_bytree", 0.5, 0.85),
            "random_state": 42
        }
        
        xgb = XGBRegressor(**param)
        model = TransformedTargetRegressor(
            regressor= xgb,
            # func=np.log1p,
            # inverse_func=np.expm1
            transformer=PowerTransformer("yeo-johnson")
        )

        
        model.fit(X_train, y_train.squeeze())
        
        cv_score = cross_val_score(
            model ,
            X_train, 
            y_train , 
            cv=10 , 
            scoring="neg_mean_absolute_error",
            n_jobs=-1
        )

        mean_score = -(cv_score.mean())
        
        mlflow.log_params(param)
        mlflow.log_metric("CV Score", mean_score)
        
        return mean_score

In [18]:
study = optuna.create_study(direction='minimize')

with mlflow.start_run(run_name="best_model"):
    # optimize the objective function
    study.optimize(objective , n_trials=10 , n_jobs=1 , show_progress_bar=True)

    # log the best params
    mlflow.log_params(study.best_params)

    # log best score
    mlflow.log_metric("best_score" , study.best_value)

    # training the XGB on best param
    best_xgb = XGBRegressor(**study.best_params , n_jobs = -1)

    best_model = TransformedTargetRegressor(
        regressor=best_xgb,
        # func=np.log1p,
        # inverse_func=np.expm1
        transformer=PowerTransformer("yeo-johnson")
    )

    best_model.fit(X_train , y_train.squeeze())

    y_pred_train = best_model.predict(X_train)
    y_pred_test = best_model.predict(X_test)

    scores = cross_val_score(
        best_model,
        X_train,
        y_train,
        scoring="neg_mean_absolute_error",
        cv=5,n_jobs=1
    )

    # logging metrics
    mlflow.log_metric("Training_error_MAE" ,mean_absolute_error(y_train ,y_pred_train))
    mlflow.log_metric("Test_error_MAE" ,mean_absolute_error(y_test ,y_pred_test))
    mlflow.log_metric("Training_error_RMSE" ,root_mean_squared_error(y_train ,y_pred_train))
    mlflow.log_metric("Test_error_RMSE" ,root_mean_squared_error(y_test ,y_pred_test))
    mlflow.log_metric("Training_r2" ,r2_score(y_train ,y_pred_train))
    mlflow.log_metric("Test_r2" ,r2_score(y_test ,y_pred_test))
    mlflow.log_metric("cross_val" , -scores.mean())

    # Generate the optuna plots
    fig_history = plot_optimization_history(study)
    fig_parallel = plot_parallel_coordinate(study)
    fig_importance = plot_param_importances(study)
    fig_slice = plot_slice(study)

    # Loginf plots
    mlflow.log_figure(fig_history, "optuna_plots/optimization_history.html")
    mlflow.log_figure(fig_importance, "optuna_plots/param_importances.html")
    mlflow.log_figure(fig_parallel, "optuna_plots/parallel_coordinate.html")
    mlflow.log_figure(fig_slice, "optuna_plots/plot_slice.html")

    # log the best model 
    mlflow.sklearn.log_model(
        sk_model=best_model, 
        name="model_xgb",
        serialization_format="cloudpickle"
    )

[I 2026-07-21 17:32:39,730] A new study created in memory with name: no-name-3336eea5-6d66-4738-bb97-ff969da8d1a6


  0%|          | 0/10 [00:00<?, ?it/s]

🏃 View run victorious-lynx-427 at: https://dagshub.com/mridul0010/NYC-Taxi-Trip-Duration.mlflow/#/experiments/9/runs/30cf56cd79a94ebd9c94cb18edeb881a
🧪 View experiment at: https://dagshub.com/mridul0010/NYC-Taxi-Trip-Duration.mlflow/#/experiments/9
[I 2026-07-21 17:36:26,358] Trial 0 finished with value: 3.056425094604492 and parameters: {'n_estimators': 789, 'max_depth': 6, 'learning_rate': 0.019548651401071218, 'subsample': 0.6083119746995082, 'min_child_weight': 41, 'gamma': 0.001579699898401299, 'reg_lambda': 13.794794710275855, 'colsample_bytree': 0.614972653505836}. Best is trial 0 with value: 3.056425094604492.
🏃 View run defiant-pig-129 at: https://dagshub.com/mridul0010/NYC-Taxi-Trip-Duration.mlflow/#/experiments/9/runs/1ce7ced3844943989cce1d9418b9a301
🧪 View experiment at: https://dagshub.com/mridul0010/NYC-Taxi-Trip-Duration.mlflow/#/experiments/9
[I 2026-07-21 17:42:52,301] Trial 1 finished with value: 2.9425482988357543 and parameters: {'n_estimators': 793, 'max_depth': 10

2026/07/21 18:13:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run best_model at: https://dagshub.com/mridul0010/NYC-Taxi-Trip-Duration.mlflow/#/experiments/9/runs/7d6ca6c040814916afd0bac74b18adde
🧪 View experiment at: https://dagshub.com/mridul0010/NYC-Taxi-Trip-Duration.mlflow/#/experiments/9


In [19]:
study.best_value

2.902940344810486

In [20]:
study.best_params

{'n_estimators': 625,
 'max_depth': 12,
 'learning_rate': 0.01817657646383304,
 'subsample': 0.7382033538889818,
 'min_child_weight': 24,
 'gamma': 0.012833011282738694,
 'reg_lambda': 6.893792887703962,
 'colsample_bytree': 0.586048838662037}

In [21]:
best_xgb = XGBRegressor(**study.best_params)

model = TransformedTargetRegressor(
        regressor=best_xgb,
        # func=np.log1p,
        # inverse_func=np.expm1
        transformer=PowerTransformer("yeo-johnson")
    )

model.fit(X_train , y_train)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.","XGBRegressor(...ree=None, ...)"
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",PowerTransformer()
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",None
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[<U38](37,)","['target_encoded__pickup_zone','target_encoded__dropoff_zone', 'target_encoded__route_time_density',...,'remainder__total_distance', 'remainder__total_travel_time','remainder__number_of_steps']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,37
regressor_ regressor_: objectFitted regressor.,XGBRegressor,"XGBRegressor(...ree=None, ...)"
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,PowerTransformer,PowerTransformer()
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None


In [22]:
y_pred = model.predict(X_test)

r2score = r2_score(y_test , y_pred)
rmse = root_mean_squared_error(y_test , y_pred)
mae = mean_absolute_error(y_test , y_pred)

print("R2 Score :-",r2score)
print("RMSE :-",rmse)
print("MAE :-",mae)

R2 Score :- 0.8048311471939087
RMSE :- 4.814977645874023
MAE :- 2.9011549949645996


In [23]:
fig_history = plot_optimization_history(study)
fig_parallel = plot_parallel_coordinate(study)
fig_importance = plot_param_importances(study)
fig_slice = plot_slice(study)

In [24]:
fig_history.show()

In [25]:
fig_parallel.show()

In [26]:
fig_importance.show()

In [27]:
fig_slice.show()